In [4]:
import pandas as pd

In [5]:
dfx = pd.read_csv('ohlcv5y.csv')
df_ohlcv = pd.DataFrame(dfx)

dfy = pd.read_csv('fund.csv')
df_fund = pd.DataFrame(dfy)

In [6]:
df_fund = df_fund[['Ticker', 'Setor', 'Indústria']].rename(columns={'Indústria': 'Industria'})
df_fund.head()

,Ticker,Setor,Industria
0,NVDA,Technology,Semiconductors
1,AAPL,Technology,Consumer Electronics
2,MSFT,Technology,Software - Infrastructure
3,AMZN,Consumer Cyclical,Internet Retail
4,GOOGL,Communication Services,Internet Content & Information


In [7]:
modelo = df_ohlcv.merge(df_fund, on='Ticker', how='left')

In [8]:
def criar_features_modelo1(modelo):

    # medias
    modelo['sma_20']         = modelo['Close'].rolling(20).mean()
    modelo['sma_50']         = modelo['Close'].rolling(50).mean()
    modelo['sma_100']        = modelo['Close'].rolling(100).mean()
    modelo['price_vs_sma20'] = modelo['Close'] / modelo['sma_20'] - 1
    modelo['price_vs_sma50'] = modelo['Close'] / modelo['sma_50'] - 1
    modelo['sma20_vs_sma50'] = modelo['sma_20'] / modelo['sma_50'] - 1

    # retornos
    modelo['return_1d']  = modelo['Close'].pct_change(1)
    modelo['return_5d']  = modelo['Close'].pct_change(5)
    modelo['return_20d'] = modelo['Close'].pct_change(20)
    modelo['return_60d'] = modelo['Close'].pct_change(60)

    # volatilidade
    modelo['volatility_10d'] = modelo['return_1d'].rolling(10).std()
    modelo['volatility_20d'] = modelo['return_1d'].rolling(20).std()

    # volume
    modelo['volume_sma20'] = modelo['Volume'].rolling(20).mean()
    modelo['volume_ratio'] = modelo['Volume'] / modelo['volume_sma20']

    return modelo


modelo = modelo.groupby('Ticker', group_keys=False).apply(criar_features_modelo1)

/var/folders/b_/p2n22yvj7wj8k1nq7fgyww940000gn/T/ipykernel_69691/1712071858.py:28: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  modelo = modelo.groupby('Ticker', group_keys=False).apply(criar_features_modelo1)


In [9]:
# analise setor

setor = (
    modelo.groupby(['Date', 'Setor'])['Close'].mean().reset_index().rename(columns={'Close': 'setor_close_avg'})
)

# setor_close_sma20
setor_close_aux = modelo.groupby(['Date', 'Setor'])['Close'].mean().reset_index().rename(columns={'Close': 'setor_close_raw'})
setor_close_aux = setor_close_aux.sort_values(['Setor', 'Date'])
setor_close_aux['setor_close_sma20'] = setor_close_aux.groupby('Setor')['setor_close_raw'].rolling(20).mean().reset_index(level=0, drop=True)
setor_close_sma20 = setor_close_aux[['Date', 'Setor', 'setor_close_sma20']]

# setor_vol20
setor_vol_aux = modelo.groupby(['Date', 'Setor'])['Volume'].mean().reset_index().rename(columns={'Volume': 'setor_vol_raw'})
setor_vol_aux = setor_vol_aux.sort_values(['Setor', 'Date'])
setor_vol_aux['setor_vol20'] = setor_vol_aux.groupby('Setor')['setor_vol_raw'].rolling(20).std().reset_index(level=0, drop=True)
setor_vol20 = setor_vol_aux[['Date', 'Setor', 'setor_vol20']]

# setor_participacao
setor_volume_total = modelo.groupby(['Date', 'Setor'])['Volume'].sum().reset_index().rename(columns={'Volume': 'setor_volume_total'})
ticker_volume = modelo.groupby(['Date', 'Ticker', 'Setor'])['Volume'].sum().reset_index().rename(columns={'Volume': 'ticker_volume'})
setor_participacao = ticker_volume.merge(setor_volume_total, on=['Date', 'Setor'], how='left')
setor_participacao['setor_participacao'] = setor_participacao['ticker_volume'] / setor_participacao['setor_volume_total']
setor_participacao = setor_participacao[['Date', 'Ticker', 'Setor', 'setor_participacao']]

# analise industria

industria = (
    modelo.groupby(['Date', 'Industria'])['Close'].mean().reset_index().rename(columns={'Close': 'industria_close_avg'})
)

# industria_close_sma20
industria_close_aux = modelo.groupby(['Date', 'Industria'])['Close'].mean().reset_index().rename(columns={'Close': 'industria_close_raw'})
industria_close_aux = industria_close_aux.sort_values(['Industria', 'Date'])
industria_close_aux['industria_close_sma20'] = industria_close_aux.groupby('Industria')['industria_close_raw'].rolling(20).mean().reset_index(level=0, drop=True)
industria_close_sma20 = industria_close_aux[['Date', 'Industria', 'industria_close_sma20']]

# industria_vol20
industria_vol_aux = modelo.groupby(['Date', 'Industria'])['Volume'].mean().reset_index().rename(columns={'Volume': 'industria_vol_raw'})
industria_vol_aux = industria_vol_aux.sort_values(['Industria', 'Date'])
industria_vol_aux['industria_vol20'] = industria_vol_aux.groupby('Industria')['industria_vol_raw'].rolling(20).std().reset_index(level=0, drop=True)
industria_vol20 = industria_vol_aux[['Date', 'Industria', 'industria_vol20']]

# industria_participacao
industria_volume_total = modelo.groupby(['Date', 'Industria'])['Volume'].sum().reset_index().rename(columns={'Volume': 'industria_volume_total'})
ticker_volume_ind = modelo.groupby(['Date', 'Ticker', 'Industria'])['Volume'].sum().reset_index().rename(columns={'Volume': 'ticker_volume_ind'})
industria_participacao = ticker_volume_ind.merge(industria_volume_total, on=['Date', 'Industria'], how='left')
industria_participacao['industria_participacao'] = industria_participacao['ticker_volume_ind'] / industria_participacao['industria_volume_total']
industria_participacao = industria_participacao[['Date', 'Ticker', 'Industria', 'industria_participacao']]

# montar sector_avg e industry_avg para merge
sector_avg = (
    setor.merge(setor_close_sma20, on=['Date', 'Setor'], how='left')
    .merge(setor_vol20, on=['Date', 'Setor'], how='left')
)

industry_avg = (
    industria
    .merge(industria_close_sma20, on=['Date', 'Industria'], how='left')
    .merge(industria_vol20, on=['Date', 'Industria'], how='left')
)

modelo = modelo.merge(sector_avg, on=['Date', 'Setor'], how='left')
modelo = modelo.merge(industry_avg, on=['Date', 'Industria'], how='left')
modelo = modelo.merge(setor_participacao, on=['Date', 'Ticker', 'Setor'], how='left')
modelo = modelo.merge(industria_participacao, on=['Date', 'Ticker', 'Industria'], how='left')

modelo['Target'] = (
    modelo.groupby('Ticker')['Close'].shift(-1) > modelo['Close']).astype(int)

modelo.head()


,Date,Ticker,Open,High,Low,Close,Volume,Setor,Industria,sma_20,...,volume_ratio,setor_close_avg,setor_close_sma20,setor_vol20,industria_close_avg,industria_close_sma20,industria_vol20,setor_participacao,industria_participacao,Target
0,2021-02-22,A,121.724499,121.927460,118.960449,119.356697,1340100.0,Healthcare,Diagnostics & Research,NaN,...,NaN,192.879015,NaN,NaN,329.473353,NaN,NaN,0.005951,0.093956,0
1,2021-02-23,A,119.366378,119.366378,116.370381,118.312943,2125400.0,Healthcare,Diagnostics & Research,NaN,...,NaN,192.515238,NaN,NaN,326.610542,NaN,NaN,0.009226,0.151358,1
2,2021-02-24,A,118.158314,121.202637,117.916702,120.825714,1898400.0,Healthcare,Diagnostics & Research,NaN,...,NaN,193.603748,NaN,NaN,329.181281,NaN,NaN,0.009684,0.143283,0
3,2021-02-25,A,120.622743,121.047984,117.839367,118.003662,1430900.0,Healthcare,Diagnostics & Research,NaN,...,NaN,190.719883,NaN,NaN,323.221949,NaN,NaN,0.006846,0.113693,0
4,2021-02-26,A,118.805819,119.366362,116.602312,117.974670,1909700.0,Healthcare,Diagnostics & Research,NaN,...,NaN,190.355273,NaN,NaN,324.367169,NaN,NaN,0.007506,0.130408,1
